In [1]:
import pandas as pd
import re
import json

In [2]:
df = pd.read_csv("/KC_PET_ACP_CTLSTT_LC_DATA_2023.csv")


In [3]:
WEEKDAY_MAP = {
    "월": "closed_mon",
    "화": "closed_tue",
    "수": "closed_wed",
    "목": "closed_thu",
    "금": "closed_fri",
    "토": "closed_sat",
    "일": "closed_sun",
}

def parse_weekdays(text):
    result = {v: 0 for v in WEEKDAY_MAP.values()}

    if not isinstance(text, str):
        return result

    # 숫자 + 월(달) 제거 → 요일 월과 충돌 방지
    text = re.sub(r"\d{1,2}월", "", text)

    for k, col in WEEKDAY_MAP.items():
        if re.search(rf"(매주\s*)?{k}(요일)?", text):
            result[col] = 1

    return result

In [4]:
import re

def normalize_week_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace(" ", "")
    text = text.replace("째주", "주")
    text = text.replace("번째주", "주")
    text = text.replace("첫째주", "1주")
    text = text.replace("둘째주", "2주")
    text = text.replace("셋째주", "3주")
    text = text.replace("넷째주", "4주")
    text = text.replace("다섯째주", "5주")

    return text


In [5]:
def parse_weeks(text):
    result = {
        "closed_week_1": 0,
        "closed_week_2": 0,
        "closed_week_3": 0,
        "closed_week_4": 0,
        "closed_week_5": 0,
    }

    if not isinstance(text, str):
        return result

    text = normalize_week_text(text)

    # "2 4 5 째주" → ["2","4","5"]
    matches = re.findall(r"(\d)\s*째주", text)
    if not matches:
        # "2 4 5 째주" 처리를 위해
        group = re.search(r"((?:\d\s*)+)\s*째주", text)
        if group:
            nums = re.findall(r"\d", group.group(1))
        else:
            nums = []
    else:
        nums = matches

    for n in nums:
        col = f"closed_week_{n}"
        if col in result:
            result[col] = 1

    if "마지막주" in text:
        result["closed_week_5"] = 1

    return result


In [6]:
import re

def parse_months(text: str):
    if not text:
        return []

    months = set()

    # 1️⃣ 연속 월 범위 (11~3월, 12~2)
    ranges = re.findall(r'(\d{1,2})\s*[~\-]\s*(\d{1,2})\s*월?', text)
    for start, end in ranges:
        start, end = int(start), int(end)

        if start <= end:
            months.update(range(start, end + 1))
        else:
            months.update(range(start, 13))
            months.update(range(1, end + 1))

    # 2️⃣ 단일 월 (1월, 12월)
    singles = re.findall(r'(?<!\d)(\d{1,2})\s*월', text)
    for m in singles:
        months.add(int(m))

    # 3️⃣ 1~12만 허용
    return sorted(m for m in months if 1 <= m <= 12)


In [7]:
tests = [
    "11~3월 매주 월요일",
    "12~2월",
    "7~8월 하절기",
    "1월 3월",
    "매주 월요일",
]

for t in tests:
    print(t, "=>", parse_months(t))


11~3월 매주 월요일 => [1, 2, 3, 11, 12]
12~2월 => [1, 2, 12]
7~8월 하절기 => [7, 8]
1월 3월 => [1, 3]
매주 월요일 => []


In [8]:
def parse_dates(text):
    if not isinstance(text, str):
        return []

    dates = re.findall(r"(\d{1,2})월\s*(\d{1,2})일", text)
    return [f"{int(m):02d}-{int(d):02d}" for m, d in dates]


In [9]:
def parse_holidays(text):
    return {
        "closed_seol": 1 if "설" in str(text) else 0,
        "closed_chuseok": 1 if "추석" in str(text) else 0,
        "closed_holiday": 1 if "법정공휴일" in str(text) else 0,
        "closed_christmas": 1 if "크리스마스" in str(text) else 0,
    }


In [10]:
records = []

for idx, row in df.iterrows():
    text = row.get("RSTDE_GUID_CN", "")

    if pd.isna(text):
        text = ""
    else:
        text = str(text)

    record = {
        "raw_text": text,
    }

    record.update(parse_weekdays(text))
    record.update(parse_weeks(text))
    record["closed_months"] = json.dumps(parse_months(text), ensure_ascii=False)
    record["closed_dates"] = json.dumps(parse_dates(text), ensure_ascii=False)
    record.update(parse_holidays(text))

    records.append(record)

result_df = pd.DataFrame(records)


In [11]:
import math

def v_int(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return 0
    return int(x)

def v_json(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    return x

def v_text(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    return str(x)


In [12]:
import pymysql
import json

conn = pymysql.connect(
    host="localhost",
    user="mini",
    password="mini",
    db="miniproject",
    charset="utf8mb4"
)

cursor = conn.cursor()

sql = """
INSERT INTO holidaydata (
  closed_mon, closed_tue, closed_wed, closed_thu,
  closed_fri, closed_sat, closed_sun,

  closed_week_1, closed_week_2, closed_week_3,
  closed_week_4, closed_week_5,

  closed_months, closed_dates,

  closed_seol, closed_chuseok,
  closed_holiday, closed_christmas,

  raw_text
)
VALUES (
  %s,%s,%s,%s,%s,%s,%s,
  %s,%s,%s,%s,%s,
  %s,%s,
  %s,%s,%s,%s,
  %s
)
"""

for _, r in result_df.iterrows():
    values = (

    v_int(r["closed_mon"]),
    v_int(r["closed_tue"]),
    v_int(r["closed_wed"]),
    v_int(r["closed_thu"]),
    v_int(r["closed_fri"]),
    v_int(r["closed_sat"]),
    v_int(r["closed_sun"]),

    v_int(r["closed_week_1"]),
    v_int(r["closed_week_2"]),
    v_int(r["closed_week_3"]),
    v_int(r["closed_week_4"]),
    v_int(r["closed_week_5"]),

    v_json(r["closed_months"]),
    v_json(r["closed_dates"]),

    v_int(r["closed_seol"]),
    v_int(r["closed_chuseok"]),
    v_int(r["closed_holiday"]),
    v_int(r["closed_christmas"]),

    v_text(r["raw_text"])
    )

    cursor.execute(sql, values)


conn.commit()
conn.close()


In [13]:
print(parse_months(text))

[]


In [14]:
print(type(record["closed_months"]), record["closed_months"])
print(type(r["closed_months"]), r["closed_months"])


<class 'str'> []
<class 'str'> []


In [ ]:
DAY_MAP = {"월":0,"화":1,"수":2,"목":3,"금":4,"토":5,"일":6}


In [19]:
import re

def parse_oper_time(text):
    """
    '월~금 09:00~18:00 토 09:00~15:00'
    """

    if text is None or str(text).strip() == "":
        return []

    result = []

    tokens = re.findall(
        r'([월화수목금토일](?:~[월화수목금토일])?)\s*([\d]{2}:[\d]{2}~[\d]{2}:[\d]{2})',
        text
    )

    for day_part, time_part in tokens:
        # 요일
        if "~" in day_part:
            s, e = day_part.split("~")
            days = list(range(DAY_MAP[s], DAY_MAP[e] + 1))
        else:
            days = [DAY_MAP[day_part]]

        # 시간
        open_t, close_t = time_part.split("~")
        is_next_day = close_t < open_t

        for d in days:
            result.append({
                "day_of_week": d,
                "open_time": open_t,
                "close_time": close_t,
                "is_next_day": is_next_day
            })

    return result


In [22]:
print(result_df.columns)

Index(['raw_text', 'closed_mon', 'closed_tue', 'closed_wed', 'closed_thu',
       'closed_fri', 'closed_sat', 'closed_sun', 'closed_week_1',
       'closed_week_2', 'closed_week_3', 'closed_week_4', 'closed_week_5',
       'closed_months', 'closed_dates', 'closed_seol', 'closed_chuseok',
       'closed_holiday', 'closed_christmas'],
      dtype='object')


In [21]:
import pymysql

conn = pymysql.connect(
    host="localhost",
    user="mini",
    password="mini",
    db="miniproject",
    charset="utf8mb4"
)

cursor = conn.cursor()

sql = """
INSERT INTO store_oper_time (
    day_of_week,
    open_time,
    close_time,
    is_next_day
)
VALUES (%s, %s, %s, %s)
"""
for _, r in result_df.iterrows():
    oper_text = r["oper_time"]

    rows = parse_oper_time(oper_text)

    for day_of_week, open_t, close_t, is_next_day in rows:
        values = (
            store_id,
            day_of_week,
            open_t,
            close_t,
            is_next_day
        )
        cursor.execute(sql, values)

conn.commit()
conn.close()

KeyError: 'oper_time'